get dataset from drive


In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# !unzip -q "/content/drive/MyDrive/Chilli-leaf-disease-dataset/Chili_Leaf_Disease_Original_Dataset.zip" -d "/content/chili_dataset"

# !ls "/content/chili_dataset"

'Chili Leaf Disease Original Dataset'


Main

In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Flatten

In [4]:
data_path = '/content/chili_dataset/Chili Leaf Disease Original Dataset'

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2) # 1,856 images

training_data = datagen.flow_from_directory(data_path, target_size=(224, 224), batch_size=32, class_mode='categorical', subset='training')

validation_data = datagen.flow_from_directory(data_path, target_size=(224, 224), batch_size=32, class_mode='categorical', subset='validation')

print(f"\nCategories found: ")
for i in training_data.class_indices.keys():
  print(i)

Found 1487 images belonging to 6 classes.
Found 369 images belonging to 6 classes.

Categories found: 
Bacterial Spot
Cercospora Leaf Spot
Curl Virus
Healthy Leaf
Nutrition Deficiency
White spot


In [5]:
vgg16_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

vgg16_base.trainable = False

vgg16_model = Sequential([
    vgg16_base,
    Flatten(),
    Dense(256, activation='relu'),
    Dense(6, activation='softmax')
])

vgg16_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
vgg16_model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,139,014 (80.64 MB)

 Trainable params: 6,424,326 (24.51 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [48]:
resnet50_base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

resnet50_base.trainable = False

resnet50_model = Sequential([
    resnet50_base,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dense(6, activation='softmax')
])

resnet50_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
resnet50_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,113,798 (91.99 MB)

 Trainable params: 526,086 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [49]:
print("--- Training VGG-16 ---")
vgg_history = vgg16_model.fit(training_data, validation_data=validation_data, epochs=15)

print("--- Training ResNet-50 ---")
resnet_history = resnet50_model.fit(training_data, validation_data=validation_data, epochs=15)

--- Training VGG-16 ---
Epoch 1/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 17s 366ms/step - accuracy: 0.9968 - loss: 0.0139 - val_accuracy: 0.7100 - val_loss: 1.0494
Epoch 2/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 17s 353ms/step - accuracy: 0.9974 - loss: 0.0202 - val_accuracy: 0.7425 - val_loss: 0.9295
Epoch 3/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 16s 344ms/step - accuracy: 0.9965 - loss: 0.0081 - val_accuracy: 0.7371 - val_loss: 0.9868
Epoch 4/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 17s 351ms/step - accuracy: 0.9974 - loss: 0.0068 - val_accuracy: 0.7263 - val_loss: 0.8987
Epoch 5/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 17s 353ms/step - accuracy: 0.9937 - loss: 0.0265 - val_accuracy: 0.7073 - val_loss: 0.9169
Epoch 6/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 17s 355ms/step - accuracy: 0.9966 - loss: 0.0074 - val_accuracy: 0.7236 - val_loss: 0.9086
Epoch 7/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 18s 392ms/step - accuracy: 0.9956 - loss: 0.0091 - val_accuracy: 0.7507 - val_loss: 0.9967
Epoch 8/15
47/47 ━━━━━━━━━━━━━━━━━━━━ 17s 357ms/step - accuracy: 0.9959 -

In [71]:
print("VGG-16 Model Results")
print(f"Final Accuracy: {vgg_history.history['accuracy'][14]*100:.2f}%")
print(f'Final loss: {vgg_history.history['loss'][14]}')
print(f'Final Accuracy with validation data: {vgg_history.history['val_accuracy'][14]*100:.2f}%')
print(f'Final loss with validation data: {vgg_history.history['val_loss'][14]}')

VGG-16 Model Results
Final Accuracy: 99.39%
Final loss: 0.008605046197772026
Final Accuracy with validation data: 69.38%
Final loss with validation data: 1.0396884679794312


VGG-16 overfitting, accuracy sama val accuracy beda jauh, kemungkinan gara" dataset terlalu kecil, mungkin kalo dataset dibanyakin pake image augmentation results will be better


In [73]:
print("Resnet-50 Model Results")
print(f'Final Accuracy: {resnet_history.history['accuracy'][14]*100:.2f}%')
print(f'Final loss: {resnet_history.history['loss'][14]}')
print(f'Final Accuracy with validation data: {resnet_history.history['val_accuracy'][14]*100:.2f}%')
print(f'Final loss with validation data: {resnet_history.history['val_loss'][14]}')

Resnet-50 Model Results
Final Accuracy: 69.13%
Final loss: 0.8118141293525696
Final Accuracy with validation data: 49.86%
Final loss with validation data: 1.4257005453109741
